In [4]:
import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [5]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}

train_df = pd.read_parquet("hf://datasets/mb7419/career-guidance-reddit/" + splits["train"])
val_df = pd.read_parquet("hf://datasets/mb7419/career-guidance-reddit/" + splits["validation"])
test_df = pd.read_parquet("hf://datasets/mb7419/career-guidance-reddit/" + splits["test"])

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [6]:
df.head()

,id,title,body,created_utc,url,retrieved_on,question_content,dominant_topic,dominant_topic_name,__index_level_0__
0,d1qr9r,How old were you when you knew what you wanted...,I think it s so dumb that we force 18 year old...,2019-09-09 13:15:22,https://www.reddit.com/r/careerguidance/commen...,2020-04-04 19:48:42,how old were you when you knew what you wanted...,0,Career Exploration and Uncertainty,6462
1,jw2ubc,Any good or simple starting point for a job?,Hello (m20) I graduated in May 2019 last...,2020-11-17 22:13:34,https://www.reddit.com/r/careerguidance/commen...,None,any good or simple starting point for a job h...,0,Career Exploration and Uncertainty,10633
2,760905,Would like to go back to school but I am unsur...,I am a 26m ive been working at a dead end job ...,2017-10-12 21:41:05,https://www.reddit.com/r/careerguidance/commen...,2017-11-11 19:28:54,would like to go back to school but i am unsur...,0,Career Exploration and Uncertainty,2348
3,j67zkc,How important is it to find your passion?,I ve never had a clear direction or vision for...,2020-10-06 16:16:54,https://www.reddit.com/r/careerguidance/commen...,None,how important is it to find your passion i ve...,0,Career Exploration and Uncertainty,10239
4,iszyh0,What should I do with my future?,I m graduating high school this year and I hav...,2020-09-15 02:35:42,https://www.reddit.com/r/careerguidance/commen...,None,what should i do with my future i m graduatin...,0,Career Exploration and Uncertainty,10038


In [7]:
df = df[['title', 'body', 'question_content', 'dominant_topic_name', 'url']]

In [8]:
df.head()

,title,body,question_content,dominant_topic_name,url
0,How old were you when you knew what you wanted...,I think it s so dumb that we force 18 year old...,how old were you when you knew what you wanted...,Career Exploration and Uncertainty,https://www.reddit.com/r/careerguidance/commen...
1,Any good or simple starting point for a job?,Hello (m20) I graduated in May 2019 last...,any good or simple starting point for a job h...,Career Exploration and Uncertainty,https://www.reddit.com/r/careerguidance/commen...
2,Would like to go back to school but I am unsur...,I am a 26m ive been working at a dead end job ...,would like to go back to school but i am unsur...,Career Exploration and Uncertainty,https://www.reddit.com/r/careerguidance/commen...
3,How important is it to find your passion?,I ve never had a clear direction or vision for...,how important is it to find your passion i ve...,Career Exploration and Uncertainty,https://www.reddit.com/r/careerguidance/commen...
4,What should I do with my future?,I m graduating high school this year and I hav...,what should i do with my future i m graduatin...,Career Exploration and Uncertainty,https://www.reddit.com/r/careerguidance/commen...


In [9]:
df.shape

(19361, 5)

In [10]:
df.isna().sum()

title                  0
body                   0
question_content       0
dominant_topic_name    0
url                    0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [13]:
# =========================================================
# CLEAN DATA
# =========================================================

required_cols = [
    "title",
    "body",
    "question_content"
]

df = df.dropna(
    subset=["question_content"]
)

for col in required_cols:

    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# =========================================================
# COMBINE TEXT
# =========================================================

df["text"] = (
    df["title"] + " " +
    df["body"] + " " +
    df["question_content"]
)


# =========================================================
# LOAD SBERT MODEL
# =========================================================

print("Loading model...")

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


# =========================================================
# POSITIVE EXAMPLES
# =========================================================

good_examples = [

    # SOFTWARE ENGINEERING
    "how to become software engineer",
    "backend developer roadmap",
    "frontend developer skills",
    "fullstack developer career path",

    # PROGRAMMING
    "coding interview preparation",
    "software development career",
    "best programming language",

    # DATA
    "data analyst career",
    "data scientist responsibilities",
    "data engineer roadmap",

    # AI / ML
    "machine learning engineer",
    "deep learning engineer",
    "artificial intelligence career",

    # CYBERSECURITY
    "cybersecurity roadmap",
    "ethical hacking career",
    "penetration tester job",

    # CLOUD / DEVOPS
    "cloud engineer roadmap",
    "devops engineer",
    "aws career",

    # NETWORKING
    "network engineer responsibilities",

    # MOBILE
    "android developer roadmap",
    "ios developer skills",

    # UI UX
    "ui ux designer roadmap",

    # GENERAL IT
    "computer science jobs",
    "software engineering growth",
    "tech industry career",
    "coding skills",
    "technical skills",
    "technology jobs"

]


# =========================================================
# NEGATIVE EXAMPLES
# =========================================================

bad_examples = [

    # EMOTIONAL
    "i feel depressed",
    "i feel anxious",
    "i feel stressed",
    "i feel lost in life",

    # RELATIONSHIP
    "my boyfriend",
    "my girlfriend",
    "my parents force me",

    # LIFE CONFUSION
    "what should i do with my life",
    "i hate my life",
    "i feel lonely",

    # NON TECH CAREER
    "work life balance",
    "best jobs for introverts",
    "college major confusion"

]


# =========================================================
# IT KEYWORDS
# =========================================================

it_keywords = [

    # SOFTWARE
    "software engineer",
    "software developer",
    "software development",

    # WEB
    "frontend",
    "backend",
    "fullstack",
    "web developer",

    # PROGRAMMING
    "programming",
    "coding",
    "computer science",

    # DATA
    "data analyst",
    "data scientist",
    "data engineer",

    # AI
    "machine learning",
    "deep learning",
    "artificial intelligence",

    # CYBERSECURITY
    "cybersecurity",
    "ethical hacking",
    "penetration testing",

    # CLOUD
    "cloud",
    "aws",
    "docker",
    "kubernetes",
    "devops",

    # MOBILE
    "android developer",
    "ios developer",

    # NETWORK
    "network engineer",

    # UI UX
    "ui ux",
    "figma",

    # TECH STACK
    "python",
    "java",
    "golang",
    "react",
    "nextjs",
    "nodejs",
    "tensorflow",
    "pytorch",
    "git",
    "github",
    "sql",
    "api"

]


# =========================================================
# NEGATIVE KEYWORDS
# =========================================================

negative_keywords = [

    "depressed",
    "depression",
    "anxiety",
    "mental health",
    "boyfriend",
    "girlfriend",
    "relationship",
    "parents",
    "lonely",
    "sad",
    "burnout",
    "emotionally"

]


# =========================================================
# FUNCTIONS
# =========================================================

def contains_it_keyword(text):

    text = text.lower()

    for keyword in it_keywords:

        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, text):
            return True

    return False


def contains_negative_keyword(text):

    text = text.lower()

    for keyword in negative_keywords:

        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, text):
            return True

    return False


# =========================================================
# ENCODE EXAMPLES
# =========================================================

print("Encoding positive examples...")

good_emb = model.encode(
    good_examples,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Encoding negative examples...")

bad_emb = model.encode(
    bad_examples,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


# =========================================================
# ENCODE DATASET
# =========================================================

texts = df["text"].tolist()

print("Encoding dataset...")

text_embs = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)


# =========================================================
# SCORING
# =========================================================

results = []

for idx, (text, emb) in enumerate(
    tqdm(
        zip(texts, text_embs),
        total=len(texts)
    )
):

    # =====================================================
    # KEYWORD BONUS
    # =====================================================

    keyword_match = contains_it_keyword(text)

    keyword_bonus = 0.08 if keyword_match else 0


    # =====================================================
    # NEGATIVE PENALTY
    # =====================================================

    negative_match = contains_negative_keyword(text)

    negative_penalty = 0.12 if negative_match else 0


    # =====================================================
    # SIMILARITY
    # =====================================================

    good_sim = cosine_similarity(
        [emb],
        good_emb
    )[0]

    bad_sim = cosine_similarity(
        [emb],
        bad_emb
    )[0]


    # =====================================================
    # TOP K
    # =====================================================

    good_topk = np.mean(
        np.sort(good_sim)[-3:]
    )

    bad_topk = np.mean(
        np.sort(bad_sim)[-3:]
    )


    # =====================================================
    # FINAL SCORE
    # =====================================================

    final_score = (
        (good_topk * 1.2)
        - (bad_topk * 0.8)
        + keyword_bonus
        - negative_penalty
    )


    # =====================================================
    # SAVE
    # =====================================================

    results.append({

        "title": df.iloc[idx]["title"],
        "body": df.iloc[idx]["body"],

        "dominant_topic_name":
            df.iloc[idx]["dominant_topic_name"],

        "url": df.iloc[idx]["url"],

        "good_score": round(float(good_topk), 4),
        "bad_score": round(float(bad_topk), 4),

        "keyword_match": keyword_match,
        "negative_match": negative_match,

        "final_score": round(float(final_score), 4)

    })


# =========================================================
# RESULT DATAFRAME
# =========================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="final_score",
    ascending=False
)


# =========================================================
# FILTER
# =========================================================

filtered_df = result_df[
    result_df["final_score"] > 0.30
]


# =========================================================
# RESULT
# =========================================================

print("\n========== RESULT ==========")

print("ORIGINAL :", len(df))
print("FILTERED :", len(filtered_df))

print("============================")


# =========================================================
# SAVE
# =========================================================

output_path = "career_it_filtered.csv"

filtered_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE!")
print("Saved:", output_path)


# =========================================================
# PREVIEW
# =========================================================

print("\nTOP RESULTS:\n")

preview_cols = [

    "title",
    "final_score",
    "keyword_match",
    "negative_match"

]

print(
    filtered_df[preview_cols].head(20)
)


Loading model...
Encoding positive examples...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]


Encoding negative examples...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.76it/s]


Encoding dataset...


100%|██████████| 19361/19361 [01:32<00:00, 209.37it/s]



========== RESULT ==========
ORIGINAL : 19361
FILTERED : 2942

DONE!
Saved: career_it_filtered.csv

TOP RESULTS:

                                                   title  final_score  \
18163              what does a data scientist really do?       0.6242   
6424   does anyone have advice for a microsoft data s...       0.6076   
18300  can someone share their interview experience f...       0.5945   
1612   what is more prominent it industry or fullstac...       0.5923   
13525    how future proof is data analysis sql skillset?       0.5825   
17718  how to ace coding challenges in assessment cen...       0.5816   
14377  what would be the best way to learn the requir...       0.5810   
2546   i want to transition into data analyst role i ...       0.5793   
18644                     tips for mobileye coding test?       0.5781   
17728  is anyone looking to become a data analyst? i ...       0.5759   
937    data analysts what are the most important exce...       0.5747   
16298  wh